In [1]:
import os, pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

# kedua file bernama report_labels.csv, jadi dibedakan dari foldernya
paths = {}
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    if "report_labels.csv" in files:
        paths[root] = os.path.join(root, "report_labels.csv")
new_p = [p for r, p in paths.items() if "relabel-32b" in r][0]
old_p = [p for r, p in paths.items() if "relabel-32b" not in r][0]
print("lama:", old_p, "\nbaru:", new_p)

old = pd.read_csv(old_p).set_index("StudyInstanceUID")[LABELS]
new = pd.read_csv(new_p).set_index("StudyInstanceUID")[LABELS]
avg = ((old.reindex(new.index) + new) / 2).fillna(new)   # studi yang kosong di label lama memakai 32B saja

def mean_auc(y, p):
    a = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5) & ~np.isnan(p[:, k])
        a.append(roc_auc_score(y[ok, k], p[ok, k]) if len(np.unique(y[ok, k])) == 2 else np.nan)
    return np.nanmean(a)

tr = pd.read_csv(f"{COMP}/train.csv")
gold = tr[tr[LABELS].notna().all(axis=1)].set_index("StudyInstanceUID")[LABELS]
for name, P in [("lama", old), ("32B", new), ("rata-rata", avg)]:
    print(f"{name:10s} mean AUC gold: {mean_auc(gold.values.astype(float), P.reindex(gold.index).values.astype(float)):.4f}")

assert set(avg.index) == set(old.index), "studi di label lama dan baru berbeda"
out = avg.reindex(old.index).reset_index()          # urutan baris sama dengan file lama
out.to_csv("/kaggle/working/report_labels.csv", index=False)
print("\nStudi:", len(out), "| sel kosong:", int(out[LABELS].isna().sum().sum()))
print("Nilai label yang muncul:", sorted(np.unique(out[LABELS].values.round(2))))

lama: /kaggle/input/datasets/kellenvalerie/rsna-knee-report-labels/report_labels.csv 
baru: /kaggle/input/notebooks/kellenvalerie/rsna-knee-01c-relabel-32b/report_labels.csv
lama       mean AUC gold: 0.8092
32B        mean AUC gold: 0.8335
rata-rata  mean AUC gold: 0.8465

Studi: 4407 | sel kosong: 0
Nilai label yang muncul: [np.float64(0.0), np.float64(0.25), np.float64(0.5), np.float64(0.75), np.float64(1.0)]
